## Data Cleaning Process

The 12 monthly datasets were combined into one dataset using Python and Pandas.

### Cleaning steps

1.  **Checked data structure**

    *   Verified that all monthly files had the same columns and format using `df.columns.equals()`.
    *   Combined the 12 files into one dataset using `pd.concat()`.

2.  **Checked data types**

    *   Converted `started_at` and `ended_at` to datetime objects using `pd.to_datetime()`.
    *   Verified data types using `all_rides_df.info()`.

3.  **Checked missing values**

    *   Identified missing values in station-related columns (`start_station_name`, `start_station_id`, `end_station_name`, `end_station_id`, `end_lat`, `end_lng`) using `all_rides_df.isnull().sum()`.
    *   No rows were dropped based on these specific missing values as they did not affect the required analysis for this notebook.

4.  **Checked duplicates**

    *   Checked `ride_id` for duplicate records using `all_rides_df.duplicated().sum()`.
    *   The check revealed no duplicate records, so no removal was necessary.

5.  **Created analysis columns**

    *   Calculated ride duration (`ride_length`) from `started_at` and `ended_at`.
    *   Extracted the day of the week (`day_of_week`) and month-based season (`season`) for time-based analysis.

6.  **Checked invalid data**

    *   Checked for invalid ride start/end times where `started_at` was after `ended_at`.
    *   Corrected these records by swapping the `started_at` and `ended_at` values to ensure chronological order.
    *   Created `ride_length_min` and `duration_group` to categorize ride lengths for further analysis, implicitly handling unreasonable durations by categorizing them into bins.

### Result

The cleaned dataset was saved to `clean_divvy_tripdata.csv` and subsequently used for the analysis phase. The cleaning process ensured that the data had a consistent structure, appropriate data types, and usable values for comparing casual riders and annual members.

# Processing Data

## Data Loading and Initial Setup

This section handles mounting Google Drive, defining project paths, and loading the individual monthly CSV files into pandas DataFrames.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Project Setup: Defining File Paths

Setting up the base project directory and subdirectories for raw and cleaned data files.

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/02_Projects/Google_DA_Certificate_Capstone_Project")

In [ ]:
RAW = PROJECT / "Data" / "2020_divvy_tripdata"

In [ ]:
CLEAN = PROJECT / "Data" / "clean_divvy_tripdata.csv"

### Importing Necessary Libraries

Importing `pandas` and `numpy` for data manipulation and numerical operations.

In [ ]:
import pandas as pd
import numpy as np

### Import 12 files for 12 months in 2022

The raw ride data for 2022 is split across 12 monthly CSV files. These files are loaded into individual DataFrames.

In [ ]:
jan_df = pd.read_csv( RAW / "202201-divvy-tripdata" / "202201-divvy-tripdata.csv")
feb_df = pd.read_csv( RAW / "202202-divvy-tripdata" / "202202-divvy-tripdata.csv")
mar_df = pd.read_csv( RAW / "202203-divvy-tripdata" / "202203-divvy-tripdata.csv")
apr_df = pd.read_csv( RAW / "202204-divvy-tripdata" / "202204-divvy-tripdata.csv")
may_df = pd.read_csv( RAW / "202205-divvy-tripdata" / "202205-divvy-tripdata.csv")
jun_df = pd.read_csv( RAW / "202206-divvy-tripdata" / "202206-divvy-tripdata.csv")
jul_df = pd.read_csv( RAW / "202207-divvy-tripdata" / "202207-divvy-tripdata.csv")
aug_df = pd.read_csv( RAW / "202208-divvy-tripdata" / "202208-divvy-tripdata.csv")
sep_df = pd.read_csv( RAW / "202209-divvy-tripdata" / "202209-divvy-publictripdata.csv")
oct_df = pd.read_csv( RAW / "202210-divvy-tripdata" / "202210-divvy-tripdata.csv")
nov_df = pd.read_csv( RAW / "202211-divvy-tripdata" / "202211-divvy-tripdata.csv")
dec_df = pd.read_csv( RAW / "202212-divvy-tripdata" / "202212-divvy-tripdata.csv")

#### Check if all datasets are compatible

In [ ]:
dfs = [jan_df, feb_df, mar_df, apr_df, may_df, jun_df, jul_df, aug_df, sep_df, oct_df, nov_df, dec_df]

all(df.columns.equals(jan_df.columns) for df in dfs)

True

#### Merge all dataset

After confirming compatibility, all individual monthly DataFrames are concatenated into a single DataFrame for comprehensive analysis.

In [ ]:
all_rides_df = pd.concat(dfs, ignore_index=True)

### Basic inspect data

Performing initial data inspection to understand the structure, data types, and a quick look at the first and last few rows.

In [ ]:
all_rides_df.tail()
all_rides_df.head()
all_rides_df.shape
all_rides_df.columns
all_rides_df.info()
all_rides_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5563947 entries, 0 to 5563946
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 551.8+ MB


,start_lat,start_lng,end_lat,end_lng
count,5.563947e+06,5.563947e+06,5.558175e+06,5.558175e+06
mean,4.190232e+01,-8.764781e+01,4.190252e+01,-8.764788e+01
std,4.618933e-02,2.939292e-02,6.836709e-02,1.092200e-01
min,4.164000e+01,-8.784000e+01,0.000000e+00,-8.814000e+01
25%,4.188103e+01,-8.766154e+01,4.188103e+01,-8.766241e+01
50%,4.190000e+01,-8.764410e+01,4.190000e+01,-8.764414e+01
75%,4.193000e+01,-8.762955e+01,4.193000e+01,-8.762963e+01
max,4.207000e+01,-8.752000e+01,4.237000e+01,0.000000e+00


In [ ]:
all_rides_df.duplicated().sum()

np.int64(0)

Checking for the presence of null values in each column, which is crucial for identifying missing data.

In [ ]:
all_rides_df.isnull().sum()

,0
ride_id,0
rideable_type,0
started_at,0
ended_at,0
start_station_name,816804
start_station_id,816804
end_station_name,874815
end_station_id,874815
start_lat,0
start_lng,0


Verifying if `ride_id` is a unique identifier for each ride.

In [ ]:
all_rides_df["ride_id"].is_unique

True

### Check for correct category

In [ ]:
all_rides_df["rideable_type"].value_counts(dropna=False)
all_rides_df["member_casual"].value_counts(dropna=False)
all_rides_df["start_station_name"].value_counts(dropna=False)
all_rides_df["end_station_name"].value_counts(dropna=False)

,count
end_station_name,
NaN,874815
Streeter Dr & Grand Ave,75134
DuSable Lake Shore Dr & North Blvd,42039
Michigan Ave & Oak St,39918
DuSable Lake Shore Dr & Monroe St,39917
...,...
Public Rack - Ashland Ave & 74th St,1
Public Rack - Avenue J & 112th St,1
Public Rack - Lawrence Ave & 103rd St,1


### Data Type Conversion

Converting `started_at` and `ended_at` columns to datetime objects for easier time-based calculations and analysis.

In [ ]:
all_rides_df["started_at"] = pd.to_datetime(all_rides_df["started_at"])
all_rides_df["ended_at"] = pd.to_datetime(all_rides_df["ended_at"])

all_rides_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667717 entries, 0 to 5667716
Data columns (total 13 columns):
 #   Column              Dtype         
---  ------              -----         
 0   ride_id             object        
 1   rideable_type       object        
 2   started_at          datetime64[ns]
 3   ended_at            datetime64[ns]
 4   start_station_name  object        
 5   start_station_id    object        
 6   end_station_name    object        
 7   end_station_id      object        
 8   start_lat           float64       
 9   start_lng           float64       
 10  end_lat             float64       
 11  end_lng             float64       
 12  member_casual       object        
dtypes: datetime64[ns](2), float64(4), object(7)
memory usage: 562.1+ MB


It's essential to ensure that the `started_at` timestamp is always chronologically before the `ended_at` timestamp. This step identifies and corrects any rides where this order is inverted.

### Find and Correct date data to make sure `started_at` is smaller than `ended_at`

In [ ]:
mask = all_rides_df["started_at"] > all_rides_df["ended_at"]

mask.sum()

all_rides_df.loc[mask]
all_rides_df.loc[mask, ["started_at", "ended_at"]] = all_rides_df.loc[mask, ["ended_at", "started_at"]].values

mask.sum()

np.int64(100)

### Create total riding time column `ride_length`

In [ ]:
all_rides_df["ride_length"] = all_rides_df["ended_at"] - all_rides_df["started_at"]

all_rides_df.head()
all_rides_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667717 entries, 0 to 5667716
Data columns (total 14 columns):
 #   Column              Dtype          
---  ------              -----          
 0   ride_id             object         
 1   rideable_type       object         
 2   started_at          datetime64[ns] 
 3   ended_at            datetime64[ns] 
 4   start_station_name  object         
 5   start_station_id    object         
 6   end_station_name    object         
 7   end_station_id      object         
 8   start_lat           float64        
 9   start_lng           float64        
 10  end_lat             float64        
 11  end_lng             float64        
 12  member_casual       object         
 13  ride_length         timedelta64[ns]
dtypes: datetime64[ns](2), float64(4), object(7), timedelta64[ns](1)
memory usage: 605.4+ MB


### Extracting Day of the Week

Creating a new column `day_of_week` from the `started_at` timestamp, representing the day of the week (Monday=0, Sunday=6).

In [ ]:
all_rides_df["day_of_week"] = all_rides_df["started_at"].dt.weekday

all_rides_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,day_of_week
0,C2F7DD78E82EC875,electric_bike,2022-01-13 11:59:47,2022-01-13 12:02:44,Glenwood Ave & Touhy Ave,525,Clark St & Touhy Ave,RP-007,42.012800,-87.665906,42.012560,-87.674367,casual,0 days 00:02:57,3
1,A6CF8980A652D272,electric_bike,2022-01-10 08:41:56,2022-01-10 08:46:17,Glenwood Ave & Touhy Ave,525,Clark St & Touhy Ave,RP-007,42.012763,-87.665967,42.012560,-87.674367,casual,0 days 00:04:21,0
2,BD0F91DFF741C66D,classic_bike,2022-01-25 04:53:40,2022-01-25 04:58:01,Sheffield Ave & Fullerton Ave,TA1306000016,Greenview Ave & Fullerton Ave,TA1307000001,41.925602,-87.653708,41.925330,-87.665800,member,0 days 00:04:21,1
3,CBB80ED419105406,classic_bike,2022-01-04 00:18:04,2022-01-04 00:33:00,Clark St & Bryn Mawr Ave,KA1504000151,Paulina St & Montrose Ave,TA1309000021,41.983593,-87.669154,41.961507,-87.671387,casual,0 days 00:14:56,1
4,DDC963BFDDA51EEA,classic_bike,2022-01-20 01:31:10,2022-01-20 01:37:12,Michigan Ave & Jackson Blvd,TA1309000002,State St & Randolph St,TA1305000029,41.877850,-87.624080,41.884621,-87.627834,member,0 days 00:06:02,3


### Add `season` column
* 1: Winter
* 2: Spring
* 3: Summer
* 4: Fall

In [ ]:
season_map = {
    12: 1, 1: 1, 2: 1,   # Winter
    3: 2, 4: 2, 5: 2,    # Spring
    6: 3, 7: 3, 8: 3,    # Summer
    9: 4, 10: 4, 11: 4   # Fall
}

all_rides_df["season"] = all_rides_df["started_at"].dt.month.map(season_map)

Displaying the first 100 rows to quickly inspect the newly created `ride_length`, `day_of_week`, and `season` columns.

In [ ]:
all_rides_df.head(100)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_length,day_of_week,season
0,C2F7DD78E82EC875,electric_bike,2022-01-13 11:59:47,2022-01-13 12:02:44,Glenwood Ave & Touhy Ave,525,Clark St & Touhy Ave,RP-007,42.012800,-87.665906,42.012560,-87.674367,casual,0 days 00:02:57,3,1
1,A6CF8980A652D272,electric_bike,2022-01-10 08:41:56,2022-01-10 08:46:17,Glenwood Ave & Touhy Ave,525,Clark St & Touhy Ave,RP-007,42.012763,-87.665967,42.012560,-87.674367,casual,0 days 00:04:21,0,1
2,BD0F91DFF741C66D,classic_bike,2022-01-25 04:53:40,2022-01-25 04:58:01,Sheffield Ave & Fullerton Ave,TA1306000016,Greenview Ave & Fullerton Ave,TA1307000001,41.925602,-87.653708,41.925330,-87.665800,member,0 days 00:04:21,1,1
3,CBB80ED419105406,classic_bike,2022-01-04 00:18:04,2022-01-04 00:33:00,Clark St & Bryn Mawr Ave,KA1504000151,Paulina St & Montrose Ave,TA1309000021,41.983593,-87.669154,41.961507,-87.671387,casual,0 days 00:14:56,1,1
4,DDC963BFDDA51EEA,classic_bike,2022-01-20 01:31:10,2022-01-20 01:37:12,Michigan Ave & Jackson Blvd,TA1309000002,State St & Randolph St,TA1305000029,41.877850,-87.624080,41.884621,-87.627834,member,0 days 00:06:02,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,82A6663F509903F5,classic_bike,2022-01-31 18:54:34,2022-01-31 19:05:01,Michigan Ave & Jackson Blvd,TA1309000002,Wabash Ave & Grand Ave,TA1307000117,41.877850,-87.624080,41.891466,-87.626761,casual,0 days 00:10:27,0,1
96,CE0530F063606F7F,electric_bike,2022-01-31 20:32:45,2022-01-31 20:41:07,Lincoln Ave & Belmont Ave,TA1309000042,Halsted St & Roscoe St,TA1309000025,41.939537,-87.668308,41.943670,-87.648950,member,0 days 00:08:22,0,1
97,1965757F99F03F3C,electric_bike,2022-01-06 12:55:16,2022-01-06 13:23:35,LaSalle St & Jackson Blvd,TA1309000004,Southport Ave & Wellington Ave,TA1307000006,41.878009,-87.630797,41.935733,-87.663576,member,0 days 00:28:19,3,1
98,569C4D7E654C5487,classic_bike,2022-01-17 11:13:14,2022-01-17 11:33:24,Sheffield Ave & Wellington Ave,TA1307000052,Kimball Ave & Belmont Ave,KA150400009X,41.936253,-87.652662,41.939398,-87.711561,member,0 days 00:20:10,0,1


### Saving Cleaned Data

The cleaned and engineered DataFrame is saved to a CSV file to avoid re-processing in future sessions.

In [ ]:
all_rides_df.to_csv(PROJECT / "Data" / "clean_divvy_tripdata.csv", index=False)

# Analyze

```markdown
# Cyclistic Bike-Share Analysis

## Business Context
Cyclistic is a fictional bike-share company used in the Google Data Analytics Capstone project. The company serves two main customer groups: casual riders, who purchase individual rides or passes, and annual members. The marketing team wants to increase the number of annual memberships because members are considered a more valuable and stable customer segment. To support this goal, this analysis examines how casual riders and annual members use the service differently.

## Business Problem
To analyze historical bike trip data to identify differences between casual riders and annual members, and to use these insights to develop strategies for converting casual riders into annual members.

## Data
The dataset consists of 12 monthly CSV files of Divvy bike-share trip data for the year 2022. Each file contains information about individual bike rides, including ride ID, bike type, start/end times, station names, geographical coordinates, and rider type (member/casual).

## Data Preparation
The 12 monthly datasets were combined into one dataset using Python and Pandas.

### Cleaning Steps

1.  **Checked data structure:**
    *   Verified that all monthly files had the same columns and format using `df.columns.equals()`.
    *   Combined the 12 files into one dataset using `pd.concat()`.
2.  **Checked data types:**
    *   Converted `started_at` and `ended_at` to datetime objects using `pd.to_datetime()`.
    *   Verified data types using `all_rides_df.info()`.
3.  **Checked missing values:**
    *   Identified missing values in station-related columns (`start_station_name`, `start_station_id`, `end_station_name`, `end_station_id`, `end_lat`, `end_lng`) using `all_rides_df.isnull().sum()`.
    *   No rows were dropped based on these specific missing values as they did not affect the required analysis for this notebook.
4.  **Checked duplicates:**
    *   Checked `ride_id` for duplicate records using `all_rides_df.duplicated().sum()`.
    *   The check revealed no duplicate records, so no removal was necessary.
5.  **Created analysis columns:**
    *   Calculated ride duration (`ride_length`) from `started_at` and `ended_at`.
    *   Extracted the day of the week (`day_of_week`) and month-based season (`season`) for time-based analysis.
6.  **Checked invalid data:**
    *   Checked for invalid ride start/end times where `started_at` was after `ended_at`.
    *   Corrected these records by swapping the `started_at` and `ended_at` values to ensure chronological order.
    *   Created `ride_length_min` and `duration_group` to categorize ride lengths for further analysis, implicitly handling unreasonable durations by categorizing them into bins.

### Result

The cleaned dataset was saved to `clean_divvy_tripdata.csv` and subsequently used for the analysis phase. The cleaning process ensured that the data had a consistent structure, appropriate data types, and usable values for comparing casual riders and annual members.

## Analysis
The analysis focused on comparing casual riders and annual members across various metrics:
*   Overall ride volume.
*   Average ride duration.
*   Ride duration variation by weekday and time of day.
*   Bike type preferences.
*   Most popular start and end stations.
*   Ride duration distribution.
*   Peak ride activity times (hour, weekday, month).

## Key Findings

### Ride Volume
![Ride Volume Comparison](visualizations/ride_volume_comparison.png)
*   Annual members generated a higher ride volume than casual riders throughout the analysis period.

### Ride Duration
![Average Ride Duration](visualizations/average_ride_duration.png)
*   Casual riders had an average ride duration that was more than twice as long as annual members. This suggests that casual riders primarily use the bikes for leisure and recreational purposes rather than transportation.

### Usage Patterns
![Usage Patterns by Time](visualizations/usage_patterns.png)
*   **Casual rider behavior**
    *   Average ride duration increased noticeably between 11:00 PM–4:00 AM and 10:00 AM–3:00 PM. Although ride volume during these periods was relatively low, riders tended to take much longer trips.
    *   Ride volume gradually increased from Thursday onward, peaked on Saturday, and remained high on Sunday. Average ride duration was also longer on weekends, especially Sunday, indicating recreational usage.
*   **Annual member behavior**
    *   Average ride duration remained relatively consistent throughout the day.
    *   Ride volume increased sharply between 3:00 PM and 7:00 PM, corresponding to typical commuting hours after work.
    *   Ride volume decreased on weekends, unlike casual riders, while ride duration remained fairly stable across the week.

### Bike Type Preference
![Bike Type Preference](visualizations/bike_type_preference.png)
*   **Bike type preference**
    *   Casual riders showed a stronger preference for electric bikes, likely because they provide a more comfortable and enjoyable riding experience.
    *   Annual members used both bike types more evenly, with no strong preference observed.

### Ride Duration Distribution
![Ride Duration Distribution](visualizations/ride_duration_distribution.png)
*   **Ride duration distribution**
    *   Casual riders were more likely to take medium- to long-duration rides.
    *   Annual members predominantly took short-duration rides.
*   **Overall Conclusion**
    *   The analysis indicates two distinct usage patterns:
        *   Casual riders primarily use the bike-sharing service for leisure and recreation. They prefer longer rides, ride more frequently on weekends and during summer, and tend to choose electric bikes for a more comfortable experience.
        *   Annual members mainly use the service for daily commuting. Their rides are shorter, more consistent in duration, and concentrated on weekday morning and late afternoon commuting hours.

## Recommendations
*   **Promote Leisure Rides**: Promote leisure rides during weekends and daytime, when casual riders are most active.
*   **Highlight Weekend Activities**: Emphasize weekend recreation and leisure activities in marketing campaigns.
*   **Showcase Electric Bikes**: Highlight the convenience and comfort of electric bikes, which are preferred by casual riders.

## Tools
*   Python
*   Pandas (for data manipulation and analysis)
*   NumPy (for numerical operations)
*   Google Colab (as the development environment)

## Project Files
*   [Jupyter Notebook](notebooks/Cyclistic_Case_Study.ipynb)
*   [Cleaned Data](data/clean_divvy_tripdata.csv)
*   [Raw Data](data/)
*   [Visualizations](visualizations/)
*   [Requirements](requirements.txt)

## Tableau Dashboard
[Link to Interactive Tableau Dashboard →]
```

To ensure we are working with the processed data, the cleaned CSV is loaded back into a DataFrame.

Question list:

1. What is the overall ride volume for casual riders compared to annual members?
2. How does average ride duration differ between casual riders and annual members?
3. How does ride duration vary by weekday and time of day across rider types?
4. How do bike type preferences differ between casual riders and annual members?
5. Which start and end stations are most popular for each rider type?
6. How does the ride duration distribution differ between casual riders and annual members? (Report)
7. When do casual riders and annual members exhibit peak ride activity (hour, weekday, and month)? (Report)

In [ ]:
all_rides_df = pd.read_csv(CLEAN)
all_rides_df.head()

##### 1. What is the overall ride volume for casual riders compared to annual members?

In [ ]:
member_volume = (
    all_rides_df
    .groupby("member_casual", as_index=False)
    .agg(member_volume=("ride_id", "count"))
)

member_volume["percentage"] = (
    member_volume["member_volume"] /
    member_volume["member_volume"].sum()
    * 100
).round(2)

member_volume

,member_casual,member_volume,percentage
0,casual,2303512,41.4
1,member,3260435,58.6


Analyzing the overall ride volume to compare casual riders with annual members.

##### 2. How does average ride duration differ between casual riders and annual members?

In [ ]:
avg_ride_duration =  (
    all_rides_df
    .groupby("member_casual", as_index=False)
    .agg(avg_ride_time=("ride_length", "mean"))
)

avg_ride_duration["avg_ride_time"] = (
    avg_ride_duration["avg_ride_time"]
    .dt.round("1s")
    .astype(str)
    .str.replace("0 days ", "")
)

avg_ride_duration

,member_casual,avg_ride_time
0,casual,00:29:08
1,member,00:12:44


Analyzing the average ride duration for casual riders and annual members.

##### 3. How does ride duration vary by weekday and time of day across rider types?

Casual riders

Analyzing the average ride duration and ride count for casual riders across different hours of the day.

In [ ]:
casual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "casual"
]

casual_duration_by_hours = (
    casual_df
    .groupby(casual_df["started_at"].dt.hour, as_index=False)
    .agg(
        avg_duration=("ride_length", "mean"),
        ride_count=("ride_id", "count")
    )
)

casual_duration_by_hours = casual_duration_by_hours.rename(columns={"started_at": "hour"})

casual_duration_by_hours["avg_duration"] = (
    casual_duration_by_hours["avg_duration"]
    .dt.round("1s")
    .astype(str)
    .str.replace("0 days ", "")
)

casual_duration_by_hours

/tmp/ipykernel_4706/1603168523.py:8: FutureWarning: A grouping was used that is not in the columns of the DataFrame and so was excluded from the result. This grouping will be included in a future version of pandas. Add the grouping as a column of the DataFrame to silence this warning.
  .agg(


,avg_duration,ride_count
0,00:30:36,46043
1,00:36:37,29760
2,00:40:54,18357
3,00:41:40,10970
4,00:38:16,7521
5,00:28:25,12276
6,00:23:59,29159
7,00:19:56,51016
8,00:19:54,69155
9,00:25:46,71511


Analyzing the average ride duration and ride count for casual riders across different days of the week.

In [ ]:
casual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "casual"
]

casual_duration_by_day_of_week = (
    casual_df
    .groupby(casual_df["started_at"].dt.weekday, as_index=False)
    .agg(
        avg_duration=("ride_length", "mean"),
        ride_count=("ride_id", "count")
    )
)

casual_duration_by_day_of_week = casual_duration_by_day_of_week.rename(columns={"started_at": "day_of_week"})

casual_duration_by_day_of_week["avg_duration"] = (
    casual_duration_by_day_of_week["avg_duration"]
    .dt.round("1s")
    .astype(str)
    .str.replace("0 days ", "")
)

casual_duration_by_day_of_week

/tmp/ipykernel_4706/628719954.py:8: FutureWarning: A grouping was used that is not in the columns of the DataFrame and so was excluded from the result. This grouping will be included in a future version of pandas. Add the grouping as a column of the DataFrame to silence this warning.
  .agg(


,avg_duration,ride_count
0,00:29:12,275246
1,00:25:53,261352
2,00:24:39,271965
3,00:25:28,306787
4,00:28:04,332242
5,00:32:34,469399
6,00:34:06,386521


Annual riders

Analyzing the average ride duration and ride count for annual members across different hours of the day.

In [ ]:
annual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "member"
]

annual_duration_by_hours = (
    annual_df
    .groupby(annual_df["started_at"].dt.hour, as_index=False)
    .agg(
        avg_duration=("ride_length", "mean"),
        ride_count=("ride_id", "count")
    )
)

annual_duration_by_hours = annual_duration_by_hours.rename(columns={"started_at": "hour"})

annual_duration_by_hours["avg_duration"] = (
    annual_duration_by_hours["avg_duration"]
    .dt.round("1s")
    .astype(str)
    .str.replace("0 days ", "")
)

annual_duration_by_hours

/tmp/ipykernel_4706/1021377507.py:8: FutureWarning: A grouping was used that is not in the columns of the DataFrame and so was excluded from the result. This grouping will be included in a future version of pandas. Add the grouping as a column of the DataFrame to silence this warning.
  .agg(


,avg_duration,ride_count
0,00:12:58,35426
1,00:13:22,21722
2,00:12:41,12560
3,00:13:15,7860
4,00:13:18,8688
5,00:10:32,30995
6,00:11:10,88634
7,00:11:38,167763
8,00:11:28,199018
9,00:11:35,140168


Analyzing the average ride duration and ride count for annual members across different days of the week.

In [ ]:
annual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "member"
]

annual_duration_by_day_of_week = (
    annual_df
    .groupby(annual_df["started_at"].dt.weekday, as_index=False)
    .agg(
        duration=("ride_length", "mean"),
        ride_count=("ride_id", "count")
    )
)

annual_duration_by_day_of_week = annual_duration_by_day_of_week.rename(columns={"started_at": "day_of_week"})

annual_duration_by_day_of_week["duration"] = (
    annual_duration_by_day_of_week["duration"]
    .dt.round("1s")
    .astype(str)
    .str.replace("0 days ", "")
)

annual_duration_by_day_of_week

/tmp/ipykernel_4706/168045623.py:8: FutureWarning: A grouping was used that is not in the columns of the DataFrame and so was excluded from the result. This grouping will be included in a future version of pandas. Add the grouping as a column of the DataFrame to silence this warning.
  .agg(


,duration,ride_count
0,00:12:18,459965
1,00:12:08,504871
2,00:12:08,511084
3,00:12:19,518250
4,00:12:33,455734
5,00:14:11,432302
6,00:14:03,378229


Comparing ride duration patterns between casual riders and annual members.

##### 4. How do bike type preferences differ between casual riders and annual members?

Casual riders

Examining the distribution of rideable types preferred by casual riders.

In [ ]:
casual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "casual"
]

casual_bike_preferences = (
    casual_df
    .groupby("rideable_type", as_index=False)
    .agg(ride_count=("ride_id", "count"))
)

casual_bike_preferences["percentage"] = (
    casual_bike_preferences["ride_count"] /
    casual_bike_preferences["ride_count"].sum()
    * 100
).round(2)

casual_bike_preferences

,rideable_type,ride_count,percentage
0,classic_bike,884485,38.40
1,docked_bike,176513,7.66
2,electric_bike,1242514,53.94


Analyzing the bike type preferences for casual riders.

Annual riders

Examining the distribution of rideable types preferred by annual members.

In [ ]:
annual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "member"
]

annual_bike_preferences = (
    annual_df
    .groupby("rideable_type", as_index=False)
    .agg(ride_count=("ride_id", "count"))
)

annual_bike_preferences["percentage"] = (
    annual_bike_preferences["ride_count"] /
    annual_bike_preferences["ride_count"].sum()
    * 100
).round(2)

annual_bike_preferences

,rideable_type,ride_count,percentage
0,classic_bike,1661662,50.96
1,electric_bike,1598773,49.04


Analyzing the bike type preferences for annual members.

##### 5. Top 10 start and end stations are most popular for each rider type?

Start station

Identifying the top 10 most frequently used start stations by casual riders.

In [ ]:
casual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "casual"
]

casual_top_10_start_station = (
    casual_df
    .groupby("start_station_name", as_index=False)
    .agg(
        ride_count=("ride_id", "count"),
        start_lat=("start_lat", "first"),
        start_lng=("start_lng", "first"),
        end_lat=("end_lat", "first"),
        end_lng=("end_lng", "first")
    )
)

casual_top_10_start_station["rank"] = (
    casual_top_10_start_station["ride_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

casual_top_10_start_station = (
    casual_top_10_start_station
    .sort_values(["rank", "ride_count"], ascending=[True, False])
)

casual_top_10_start_station

,start_station_name,ride_count,start_lat,start_lng,end_lat,end_lng,rank
1440,Streeter Dr & Grand Ave,57980,41.892278,-87.612043,41.891023,-87.635480,1
345,DuSable Lake Shore Dr & Monroe St,31765,41.880958,-87.616743,41.867226,-87.615355,2
775,Millennium Park,25451,41.881129,-87.624140,41.883984,-87.624684,3
768,Michigan Ave & Oak St,25233,41.900960,-87.623777,41.926277,-87.630834,4
346,DuSable Lake Shore Dr & North Blvd,23635,41.911722,-87.626804,41.944540,-87.654678,5
...,...,...,...,...,...,...,...
1462,Tuley (Murray) Park,1,41.730000,-87.610000,41.740000,-87.600000,729
1473,Vincennes Ave & 95th Pl,1,41.720000,-87.650000,41.720000,-87.640000,729
1522,Wentworth Ave & 79th St,1,41.750000,-87.630000,41.750000,-87.630000,729
1538,Western Ave & 116th St,1,41.680000,-87.680000,41.680000,-87.680000,729


Identifying the top 10 most frequently used start stations by annual members.

In [ ]:
annual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "member"
]

annual_top_10_start_station = (
    annual_df
    .groupby("start_station_name", as_index=False)
    .agg(
    ride_count=("ride_id", "count"),
    start_lat=("start_lat", "first"),
    start_lng=("start_lng", "first"),
    end_lat=("end_lat", "first"),
    end_lng=("end_lng", "first")
    )
)

annual_top_10_start_station["rank"] = (
    annual_top_10_start_station["ride_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

annual_top_10_start_station = (
    annual_top_10_start_station
    .sort_values(["rank", "ride_count"], ascending=[True, False])
)

annual_top_10_start_station

,start_station_name,ride_count,start_lat,start_lng,end_lat,end_lng,rank
566,Kingsbury St & Kinzie St,23963,41.889177,-87.638506,41.897448,-87.628722,1
222,Clark St & Elm St,21325,41.902973,-87.631280,41.907993,-87.631501,2
1413,Wells St & Concord Ln,20773,41.912003,-87.634631,41.893808,-87.641697,3
1371,University Ave & 57th St,19354,41.791478,-87.599861,41.793242,-87.587782,4
249,Clinton St & Washington Blvd,19198,41.883629,-87.641295,41.894722,-87.634362,5
...,...,...,...,...,...,...,...
1421,Wentworth Ave & 103rd St,1,41.710000,-87.630000,41.721850,-87.622854,707
1427,Wentworth Ave & 79th St,1,41.750000,-87.630000,41.720000,-87.630000,707
1464,Whippie St & 26th St,1,41.840000,-87.700000,41.840000,-87.710000,707
1467,William Rainey Harper High School,1,41.770000,-87.670000,41.779835,-87.634774,707


Summarizing findings about the most popular start stations for both casual and annual riders.

End station

Identifying the top 10 most frequently used end stations by casual riders.

In [ ]:
casual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "casual"
]

casual_top_10_end_station = (
    casual_df
    .groupby("end_station_name", as_index=False)
        .agg(
        ride_count=("ride_id", "count"),
        start_lat=("start_lat", "first"),
        start_lng=("start_lng", "first"),
        end_lat=("end_lat", "first"),
        end_lng=("end_lng", "first")
    )
)

casual_top_10_end_station["rank"] = (
    casual_top_10_end_station["ride_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

casual_top_10_end_station = (
    casual_top_10_end_station
    .sort_values(["rank", "ride_count"], ascending=[True, False])
)

casual_top_10_end_station

,end_station_name,ride_count,start_lat,start_lng,end_lat,end_lng,rank
1457,Streeter Dr & Grand Ave,59766,41.795203,-87.580957,41.892278,-87.612043,1
347,DuSable Lake Shore Dr & Monroe St,29525,41.865293,-87.617884,41.880958,-87.616743,2
780,Millennium Park,26560,41.877847,-87.623966,41.881032,-87.624084,3
773,Michigan Ave & Oak St,26401,41.926277,-87.630834,41.900960,-87.623777,4
348,DuSable Lake Shore Dr & North Blvd,26122,41.793242,-87.587782,41.911722,-87.626804,5
...,...,...,...,...,...,...,...
1542,West Chatham Park,1,41.730000,-87.550000,41.750000,-87.630000,722
1543,West Lawn Park,1,41.800000,-87.710000,41.770000,-87.730000,722
1550,Western Ave & 106th St - West,1,41.690000,-87.670000,41.700000,-87.680000,722
1584,William Rainey Harper High School,1,41.780000,-87.670000,41.780000,-87.670000,722


Identifying the top 10 most frequently used end stations by annual members.

In [ ]:
casual_df = all_rides_df.loc[
    all_rides_df["member_casual"] == "member"
]

annual_top_10_end_station = (
    casual_df
    .groupby("end_station_name", as_index=False)
        .agg(
        ride_count=("ride_id", "count"),
        start_lat=("start_lat", "first"),
        start_lng=("start_lng", "first"),
        end_lat=("end_lat", "first"),
        end_lng=("end_lng", "first")
    )
)

annual_top_10_end_station["rank"] = (
    annual_top_10_end_station["ride_count"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

annual_top_10_end_station = (
    annual_top_10_end_station
    .sort_values(["rank", "ride_count"], ascending=[True, False])
)

annual_top_10_end_station

,end_station_name,ride_count,start_lat,start_lng,end_lat,end_lng,rank
563,Kingsbury St & Kinzie St,23759,41.884521,-87.627983,41.889177,-87.638506,1
224,Clark St & Elm St,21689,41.911282,-87.638517,41.902973,-87.631280,2
1396,Wells St & Concord Ln,21351,41.901315,-87.677409,41.912133,-87.634656,3
1356,University Ave & 57th St,19879,41.795212,-87.580715,41.791478,-87.599861,4
251,Clinton St & Washington Blvd,19847,41.877935,-87.644040,41.883380,-87.641170,5
...,...,...,...,...,...,...,...
1447,Whippie St & 26th St,1,41.844524,-87.702040,41.840000,-87.700000,705
1450,William Rainey Harper High School,1,41.780000,-87.670000,41.780000,-87.670000,705
1456,Winchester Ave & 87th St,1,41.810000,-87.670000,41.740000,-87.670000,705
1460,Wolcott Ave & 61st St,1,41.876704,-87.639578,41.780000,-87.670000,705


Summarizing findings about the most popular end stations for both casual and annual riders.

In [ ]:
output_file_path = PROJECT / "Cyclistic_Analysis_Q5.xlsx"

# DataFrames to export for Question 5
q5_dataframes_to_export = {
    "Q5_Casual_Top_Start_Stations": casual_top_10_start_station,
    "Q5_Casual_Top_End_Stations": casual_top_10_end_station,
    "Q5_Annual_Top_Start_Stations": annual_top_10_start_station,
    "Q5_Annual_Top_End_Stations": annual_top_10_end_station
}

# Export the selected Question 5 dataframes to the Excel file
with pd.ExcelWriter(output_file_path) as writer:
    for sheet_name, df in q5_dataframes_to_export.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Question 5 data (top start/end stations for casual/annual riders) exported to: {output_file_path}")
print("Each dataframe is on its own sheet, ready for Tableau map charts using 'start_lat', 'start_lng', 'end_lat', 'end_lng' fields.")

Question 5 data (top start/end stations for casual/annual riders) exported to: /content/drive/MyDrive/02_Projects/Google_DA_Certificate_Capstone_Project/Cyclistic_Analysis_Q5.xlsx
Each dataframe is on its own sheet, ready for Tableau map charts using 'start_lat', 'start_lng', 'end_lat', 'end_lng' fields.


##### 6. How does the ride duration distribution differ between casual riders and annual members?

Analyzing the distribution of ride durations for both casual and annual members by categorizing ride lengths into predefined bins.

In [ ]:
all_rides_df["ride_length"] = pd.to_timedelta(
    all_rides_df["ride_length"]
)

all_rides_df["ride_length_min"] = (
    all_rides_df["ride_length"]
    .dt.total_seconds() / 60
)

bins = [0, 5, 10, 15, 20, 30, 60, float("inf")]

labels = [
    "0-5",
    "5-10",
    "10-15",
    "15-20",
    "20-30",
    "30-60",
    "60+"
]

all_rides_df["duration_group"] = pd.cut(
    all_rides_df["ride_length_min"],
    bins=bins,
    labels=labels
)

duration_distribution = (
    all_rides_df
    .groupby(
        ["member_casual", "duration_group"],
        as_index=False
    )
    .agg(
        ride_count=("ride_id", "count")
    )
)

duration_distribution

/tmp/ipykernel_4706/2534410254.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(


,member_casual,duration_group,ride_count
0,casual,0-5,304681
1,casual,5-10,568030
2,casual,10-15,425827
3,casual,15-20,273287
4,casual,20-30,307245
5,casual,30-60,278280
6,casual,60+,145957
7,member,0-5,785574
8,member,5-10,1037980
9,member,10-15,593666


Analyzing the ride duration distribution for casual riders and annual members.

##### 7. When do casual riders and annual members exhibit peak ride activity (hour, weekday, and month)?

Preparing `started_at` and `ended_at` columns for time-based analysis.

In [ ]:
all_rides_df["started_at"] = pd.to_datetime(all_rides_df["started_at"])

all_rides_df["ended_at"] = pd.to_datetime(all_rides_df["ended_at"])

In [ ]:
# Peak activity by hour for casual riders
peak_casual_hour = (
    all_rides_df[all_rides_df['member_casual'] == 'casual']
    .groupby(all_rides_df['started_at'].dt.hour)
    .size()
    .sort_values(ascending=False)
)
print("Peak Hours for Casual Riders:")
print(peak_casual_hour.head())

Peak Hours for Casual Riders:
started_at
17    218627
18    196224
16    196165
15    176726
14    158578
dtype: int64


Analyzing peak ride activity by weekday for casual riders.

In [ ]:
# Peak activity by weekday for casual riders
peak_casual_weekday = (
    all_rides_df[all_rides_df['member_casual'] == 'casual']
    .groupby(all_rides_df['started_at'].dt.day_name())
    .size()
    .sort_values(ascending=False)
)
print("\nPeak Weekdays for Casual Riders:")
print(peak_casual_weekday.head())


Peak Weekdays for Casual Riders:
started_at
Saturday    469399
Sunday      386521
Friday      332242
Thursday    306787
Monday      275246
dtype: int64


Analyzing peak ride activity by month for casual riders.

In [ ]:
# Peak activity by month for casual riders
peak_casual_month = (
    all_rides_df[all_rides_df['member_casual'] == 'casual']
    .groupby(all_rides_df['started_at'].dt.month_name())
    .size()
    .sort_values(ascending=False)
)
print("\nPeak Months for Casual Riders:")
peak_casual_month


Peak Months for Casual Riders:


,0
started_at,
July,406055
June,369051
August,358924
September,296697
May,280415
October,208989
April,126417
November,100772
March,89882


Analyzing peak ride activity by hour for annual members.

In [ ]:
# Peak activity by hour for annual members
peak_member_hour = (
    all_rides_df[all_rides_df['member_casual'] == 'member']
    .groupby(all_rides_df['started_at'].dt.hour)
    .size()
    .sort_values(ascending=False)
)
print("\nPeak Hours for Annual Members:")
print(peak_member_hour.head())


Peak Hours for Annual Members:
started_at
17    340694
16    283760
18    278137
15    215162
19    201815
dtype: int64


Analyzing peak ride activity by weekday for annual members.

In [ ]:
# Peak activity by weekday for annual members
peak_member_weekday = (
    all_rides_df[all_rides_df['member_casual'] == 'member']
    .groupby(all_rides_df['started_at'].dt.day_name())
    .size()
    .sort_values(ascending=False)
)
print("\nPeak Weekdays for Annual Members:")
print(peak_member_weekday.head())


Peak Weekdays for Annual Members:
started_at
Thursday     518250
Wednesday    511084
Tuesday      504871
Monday       459965
Friday       455734
dtype: int64


Analyzing peak ride activity by month for annual members.

In [ ]:
# Peak activity by month for annual members
peak_member_month = (
    all_rides_df[all_rides_df['member_casual'] == 'member']
    .groupby(all_rides_df['started_at'].dt.month_name())
    .size()
    .sort_values(ascending=False)
)
print("\nPeak Months for Annual Members:")
peak_member_month


Peak Months for Annual Members:


,0
started_at,
August,427008
July,417433
September,404642
June,400153
May,354443
October,349696
April,244832
November,236963
March,194160


Summarizing peak ride activity patterns for casual riders and annual members.

## Export all results to one spreadsheet


In [ ]:
# Define the output path for the Excel file
output_file_path = PROJECT / "Cyclistic_Analysis.xlsx"

# Convert Q7 Series results to DataFrames for export
peak_casual_hour_df = peak_casual_hour.to_frame(name='ride_count').reset_index().rename(columns={'started_at': 'hour'})
peak_casual_weekday_df = peak_casual_weekday.to_frame(name='ride_count').reset_index().rename(columns={'started_at': 'day_of_week'})
peak_casual_month_df = peak_casual_month.to_frame(name='ride_count').reset_index().rename(columns={'started_at': 'month'})

peak_member_hour_df = peak_member_hour.to_frame(name='ride_count').reset_index().rename(columns={'started_at': 'hour'})
peak_member_weekday_df = peak_member_weekday.to_frame(name='ride_count').reset_index().rename(columns={'started_at': 'day_of_week'})
peak_member_month_df = peak_member_month.to_frame(name='ride_count').reset_index().rename(columns={'started_at': 'month'})

# Create a dictionary of all results to export with unique sheet names
# Note: For Q5, only annual top stations are exported as casual data was overwritten in previous steps.
all_results = {
    "Q1_Ride_Volume": member_volume,
    "Q2_Avg_Duration": avg_ride_duration,
    "Q3_Casual_Hour_Duration": casual_duration_by_hours,
    "Q3_Casual_Weekday_Duration": casual_duration_by_day_of_week,
    "Q3_Annual_Hour_Duration": annual_duration_by_hours,
    "Q3_Annual_Weekday_Duration": annual_duration_by_day_of_week,
    "Q4_Casual_Bike_Type": casual_bike_preferences,
    "Q4_Annual_Bike_Type": annual_bike_preferences,
    "Q5_Annual_Top_Start_Stations": top_10_start_station, # This variable currently holds annual data
    "Q5_Annual_Top_End_Stations": top_10_end_station,     # This variable currently holds annual data
    "Q6_Duration_Distribution": duration_distribution,
    "Q7_Casual_Peak_Hour": peak_casual_hour_df,
    "Q7_Casual_Peak_Weekday": peak_casual_weekday_df,
    "Q7_Casual_Peak_Month": peak_casual_month_df,
    "Q7_Annual_Peak_Hour": peak_member_hour_df,
    "Q7_Annual_Peak_Weekday": peak_member_weekday_df,
    "Q7_Annual_Peak_Month": peak_member_month_df
}

# Write all results to a single Excel file with multiple sheets
with pd.ExcelWriter(output_file_path) as writer:
    for sheet_name, df in all_results.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"All analysis results have been exported to: {output_file_path}")

All analysis results have been exported to: /content/drive/MyDrive/02_Projects/Google_DA_Certificate_Capstone_Project/Cyclistic_Analysis.xlsx
